# Imports

In [63]:
import numpy as np
from scipy.optimize import minimize
from scipy.optimize import NonlinearConstraint
import seaborn as sns 
import matplotlib.pyplot as plt

# Data Loading

In [ ]:
data_folder_path = "../../../../connect_tome/data/flywire_fafb"
connections_path = f'{data_folder_path}/connections_princeton_no_threshold.csv.gz'
cell_types_path = f'{data_folder_path}/consolidated_cell_types.csv.gz'

# Model

$\begin{gather}
\tau_x \dot{y}_i + y_i = x_i + \sum_j W_{ij}z_j \\
\tau_y \dot{z_j} + z_j = \sum_i M_{ji}y_i \\
y^{\infty} = x^{\infty} + Wz^{\infty} \\
y^{\infty} = x^{\infty} + WMy^{\infty} \\
y^{\infty} = (I - WM)^{-1}x^{\infty} 
\end{gather}$

Assume $\langle x \rangle = 0$.

In [55]:
# First pass, weight matrix is a rank 1 outer product. Minimizing covariance. 

def model(input_data, w, m):
    """Gets, X, W, M and returns Y in steady state.

    Args:
        input_data: neurons x samples 
        w: neurons x 1 
        m: 1 x neurons 

    Returns:
        y: neurons x neurons
    
    """
    state_dim = input_data.shape[0]
    matrix = np.eye(state_dim) - w @ m
    return np.linalg.solve(matrix, input_data)

def residual(params, input_data):
    """Calculates covariance residual
    
    Args:
        params: (2*neurons,)
        input_data: neurons x neurons
    
    Returns:
        residuals: (neurons**2,)
    """
    state_dim = input_data.shape[0]

    w = params[:state_dim, np.newaxis] # (neurons, 1)
    m = params[np.newaxis, state_dim:] # (1, neurons)

    outputs = model(input_data, w, m)
    output_cov = np.cov(outputs) # (neurons, neurons)

    return np.eye(state_dim) - output_cov

def obj_fun(params, input_data):
    return np.linalg.norm(residual(params, input_data))

def cons_params(params):
    state_dim = len(params) // 2
    w, m = params[:state_dim], params[state_dim:]
    return (w @ m) 


In [ ]:
# Synthetic Data - first assume activations are drawn from standard gaussian 
n_samples = 5000
n_neurons = 14
x = np.random.standard_normal((n_neurons, n_samples))
sns.heatmap(np.cov(x), annot=True, fmt=".2f")

In [ ]:
# Initialization
init_w = np.repeat([0.2, 0.5], n_neurons // 2)
init_m = np.full(n_neurons, 0.9)
init_params = np.concatenate([init_w / np.linalg.norm(init_w), init_m / np.linalg.norm(init_m)])

In [56]:
# Unbounded optimization 
nonlin_const = NonlinearConstraint(cons_params, -np.inf, 1-1e-3)
res = minimize(obj_fun, init_params, args=(x), constraints=[nonlin_const])

In [68]:
res.x[:n_neurons], res.x[n_neurons:]

(array([-4.98447464, -5.1126974 , -4.67282375, -4.67444126, -5.11845455,
        -4.35763271, -4.9320768 ,  3.11195778,  2.08348824,  2.87203797,
         2.58511102,  2.3540204 ,  2.82000287,  2.13874906]),
 array([ 4.19740774,  4.17961665,  4.84277905,  4.53283584,  4.16602125,
         5.14592945,  4.2497462 , -2.47297547, -3.67850977, -3.02536364,
        -3.13528214, -3.43251284, -2.88306071, -3.58981299]))

In [70]:
# Unbounded optimization 
from scipy.optimize import least_squares

def residual(params, x):
    n = x.shape[0]
    w, v = params[:n], params[n:]
    y = x + np.outer(w, v @ x)          # (I + w vᵀ) x
    return (np.eye(n) - np.cov(y)).ravel()

res = least_squares(residual, init_params, args=(x,), method="trf")

w, v = res.x[:n_neurons], res.x[n_neurons:]
denom = 1 + v @ w                        # equals 1 / (1 - m·w)
assert denom > 0, "solution is in the m·w > 1 branch"
m = v / denom
s = m @ w                                # guaranteed != 1 for finite v


np.float64(-0.047393515508798674)